# M3T3 - Consumidor Spark Structured Streaming

Este cuaderno lee los eventos generados por `M3T3_productor_eventos_streaming.ipynb`, calcula métricas por ventana, detecta alertas simples y escribe resultados en sinks Parquet.

Incluye ejemplos de watermarks, checkpoints, salida raw, salida agregada y uso de `foreachBatch` para métricas por microbatch.

In [ ]:
import os
import shutil

INPUT_DIR = os.path.abspath("input/m3t3_events")
RAW_OUT = os.path.abspath("output/m3t3_raw_parquet")
CLICK_METRICS_OUT = os.path.abspath("output/m3t3_click_metrics_parquet")
ALERTS_OUT = os.path.abspath("output/m3t3_alerts_parquet")
BATCH_METRICS_OUT = os.path.abspath("output/m3t3_batch_metrics")
CHK_RAW = os.path.abspath("checkpoint/m3t3_raw")
CHK_CLICKS = os.path.abspath("checkpoint/m3t3_click_metrics")
CHK_ALERTS = os.path.abspath("checkpoint/m3t3_alerts")
CHK_BATCH = os.path.abspath("checkpoint/m3t3_batch_metrics")

for path in (RAW_OUT, CLICK_METRICS_OUT, ALERTS_OUT, BATCH_METRICS_OUT, CHK_RAW, CHK_CLICKS, CHK_ALERTS, CHK_BATCH):
    shutil.rmtree(path, ignore_errors=True)

os.makedirs(INPUT_DIR, exist_ok=True)
print("Entrada:", INPUT_DIR)

In [ ]:
# Descomenta si necesitas instalar PySpark
# %pip install pyspark==3.5.1

In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .appName("M3T3PipelineStreaming")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate())

spark.sparkContext.setLogLevel("WARN")
spark

## Esquema y lectura streaming

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType
from pyspark.sql.functions import col, current_timestamp, window, count, approx_count_distinct, sum as spark_sum, lit

schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("ts", TimestampType(), True),
    StructField("userId", StringType(), True),
    StructField("url", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("cardId", StringType(), True),
    StructField("merchant", StringType(), True),
    StructField("source", StringType(), True)
])

events = (spark.readStream
    .schema(schema)
    .json(INPUT_DIR)
    .withColumn("processing_time", current_timestamp()))

valid_events = events.dropna(subset=["event_id", "event_type", "ts"])
valid_events.printSchema()

## Sink raw en Parquet

In [ ]:
raw_query = (valid_events.writeStream
    .format("parquet")
    .option("path", RAW_OUT)
    .option("checkpointLocation", CHK_RAW)
    .outputMode("append")
    .start())

raw_query

## Métricas de clickstream con ventanas y watermark

In [ ]:
clicks = valid_events.filter(col("event_type") == "click")

click_metrics = (clicks
    .withWatermark("ts", "10 minutes")
    .groupBy(window(col("ts"), "2 minutes"), col("url"))
    .agg(
        count("*").alias("num_clicks"),
        approx_count_distinct("userId").alias("usuarios_unicos")
    ))

click_query = (click_metrics.writeStream
    .format("parquet")
    .option("path", CLICK_METRICS_OUT)
    .option("checkpointLocation", CHK_CLICKS)
    .outputMode("append")
    .start())

click_query

## Detección simple de alertas por tarjeta

In [ ]:
transactions = valid_events.filter(col("event_type") == "transaction")

alerts = (transactions
    .withWatermark("ts", "10 minutes")
    .groupBy(window(col("ts"), "2 minutes", "30 seconds"), col("cardId"))
    .agg(count("*").alias("num_tx"), spark_sum("amount").alias("amount_total"))
    .filter((col("num_tx") >= 4) | (col("amount_total") >= 900))
    .withColumn("rule", lit("TX_BURST_OR_AMOUNT")))

alerts_query = (alerts.writeStream
    .format("parquet")
    .option("path", ALERTS_OUT)
    .option("checkpointLocation", CHK_ALERTS)
    .outputMode("append")
    .start())

alerts_query

## foreachBatch para documentar métricas por microbatch

Este ejemplo escribe un pequeño resumen por microbatch usando lógica batch. Debe ser idempotente si se adapta a producción.

In [ ]:
from pyspark.sql.functions import count as spark_count

def write_batch_metrics(batch_df, batch_id):
    metrics = (batch_df
        .groupBy("event_type")
        .agg(spark_count("*").alias("records"))
        .withColumn("batch_id", lit(batch_id)))

    (metrics.write
        .mode("append")
        .parquet(BATCH_METRICS_OUT))

batch_metrics_query = (valid_events.writeStream
    .foreachBatch(write_batch_metrics)
    .option("checkpointLocation", CHK_BATCH)
    .start())

batch_metrics_query

## Monitorización rápida

In [ ]:
[(q.name, q.id, q.status) for q in spark.streams.active]

In [ ]:
raw_query.lastProgress

## Parada ordenada

In [ ]:
for query in spark.streams.active:
    print("Deteniendo", query.id)
    query.stop()

spark.streams.active